# 01 — Bronze Ingest

Clones the repo, generates synthetic GGM-aligned traffic & water data, and writes it as Delta tables in the **Bronze** layer of the attached Lakehouse.

**Before running:** attach the `FabricIQ` Lakehouse via the left-hand panel.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faker", "pandas", "pyarrow"], check=True)

In [ ]:
import subprocess, os
REPO = '/tmp/fabric-iq'
if not os.path.exists(REPO):
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/AjinkyaSoDev/fabric-iq.git', REPO],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
else:
    print('Repo already present at', REPO)

In [ ]:
import sys
sys.path.insert(0, '/tmp/fabric-iq')

from synth.generators import traffic, water
from datetime import datetime, timedelta, timezone

DAYS = 14
start = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0) - timedelta(days=DAYS)

datasets = {}
datasets.update(traffic.gen_all_traffic(start, days=DAYS))
datasets.update(water.gen_all_water(start, days=DAYS))

for name, df in datasets.items():
    print(f'  {name}: {len(df):>7} rows')

In [ ]:
for name, df in datasets.items():
    table_name = f'bronze_{name.lower()}'
    spark_df = spark.createDataFrame(df)
    spark_df.write.mode('overwrite').format('delta').saveAsTable(table_name)
    print(f'  Wrote {table_name}: {len(df)} rows')

print('\n✅ Bronze layer complete!')